# 지도 학습
학습·테스트를 분할하고 스케일러/PCA/교차검증은 학습 데이터에서만 적합합니다.

명세: [코드별 문서](../docs/MODULE_SPECS.md) · [구조](../docs/ARCHITECTURE.md)

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
roots = [cwd, cwd.parent]
project_root = next((p for p in roots if (p / 'src/MachineLearning/common.py').exists()), None)
if project_root is None:
    raise RuntimeError('저장소 루트 또는 codebook 폴더에서 실행하세요.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
from sklearn.datasets import load_iris, load_diabetes
X, y = load_iris(return_X_y=True)
R, target = load_diabetes(return_X_y=True)


In [ ]:
from src.MachineLearning.Linear.least_squares import run_least_squares
from src.MachineLearning.Linear.lasso import LassoProfile, run_lasso
from src.MachineLearning.Linear.ridge import RidgeProfile, run_ridge
for name, runner, profile in [('OLS', run_least_squares, None), ('Lasso', run_lasso, LassoProfile()), ('Ridge', run_ridge, RidgeProfile(param_grid={'alpha': [0.1, 1, 10]}))]:
    result = runner(R, target, profile)
    print(name, result.test_metrics, result.best_params)


In [ ]:
from src.MachineLearning.Linear.logistic import run_logistic
from src.MachineLearning.Trees.decision_tree import run_decision_tree
from src.MachineLearning.Trees.random_forest import run_random_forest
from src.MachineLearning.Trees.gradient_boosting import GradientBoostingProfile, run_gradient_boosting
from src.MachineLearning.Trees.adaboost import run_adaboost
from src.MachineLearning.Neighbors.knn import run_knn
from src.MachineLearning.SVM.svm import run_svm
from src.MachineLearning.NeuralNetworks.mlp import MLPProfile, run_mlp
from src.MachineLearning.NaiveBayes.naive_bayes import run_naive_bayes
models = [('Logistic', run_logistic, None), ('DT', run_decision_tree, None), ('RF', run_random_forest, None), ('GB', run_gradient_boosting, GradientBoostingProfile(task='classification')), ('AdaBoost', run_adaboost, None), ('KNN', run_knn, None), ('SVM', run_svm, None), ('MLP', run_mlp, MLPProfile(solver='lbfgs', max_iter=2000)), ('NB', run_naive_bayes, None)]
for name, runner, profile in models:
    result = runner(X, y, profile)
    print(name, result.test_metrics)


In [ ]:
from src.MachineLearning.Decomposition.pca import PCAClassificationProfile, run_pca_classification
result = run_pca_classification(X, y, PCAClassificationProfile(param_grid={'pca__n_components': [2, 3], 'C': [0.1, 1]}, cv=3))
print(result.test_metrics, result.best_params)
print('새 데이터 예측:', result.model.predict(X[:3]))


In [ ]:
from src.MachineLearning.Linear.numerical import NumericalProfile, run_numerical_regression
for method in ['normal', 'svd', 'tsvd', 'ridge']:
    result = run_numerical_regression(R, target, NumericalProfile(method=method))
    print(method, result.rank, result.condition_number)
